In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import timm
from pathlib import Path
from PIL import Image
from torchvision import transforms
import os


# Load Model Function

def load_model(pt_path, device="cpu"):
    # Load checkpoint
    checkpoint = torch.load(pt_path, map_location=device)

    # Load classes + class_to_idx
    classes = checkpoint["classes"]   # e.g. ["Email", "Invoice", "Receipt"]
    class_to_idx = checkpoint["class_to_idx"]  # e.g. {"Email":0, "Invoice":1, "Receipt":2}
    num_classes = len(classes)

    # Recreate model
    model = timm.create_model("mobilenetv3_small_100", pretrained=False, num_classes=num_classes)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()

    return model, classes, class_to_idx



# Preprocess Function

def preprocess_image(img_path, img_size=224):
    tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])
    img = Image.open(img_path).convert("RGB")
    return tf(img).unsqueeze(0)  # Add batch dimension


# Prediction Function

def predict_image(model, img_path, classes, class_to_idx, device="cpu", img_size=224):
    x = preprocess_image(img_path, img_size).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.nn.functional.softmax(logits, dim=1)  # convert to probabilities
        conf, pred_idx = torch.max(probs, 1)  # best class + confidence

    # Ensure correct mapping: idx → class
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    pred_class = idx_to_class[pred_idx.item()]

    return pred_class, pred_idx.item(), conf.item()



# Run Inference

if __name__ == "__main__":
    # ---- Update these paths ----
    pt_path = r"D:\MobileNetV3_Small_Model\outputs\best_mnv3.pt"
    img_dir = r"D:\MobileNetV3_Small_Model\dataset_root\test\doc"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load model
    model, classes, class_to_idx = load_model(pt_path, device)

    # Loop through all files in the directory
    for root, _, files in os.walk(img_dir):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):  # only image files
                img_path = os.path.join(root, file)

                try:
                    pred_class, pred_idx, confidence = predict_image(model, img_path, classes, class_to_idx, device)
                    print(f"Prediction for {file}: {pred_class} (index={pred_idx}, confidence={confidence:.4f})")
                except Exception as e:
                    print(f"Error processing {file}: {e}")